In [ ]:
from pathlib import Path

def get_mp4_stems(folder_path, recursive=True):
    folder = Path(folder_path)
    files = folder.rglob("*.mp4") if recursive else folder.glob("*.mp4")
    return [file.stem for file in files]

def get_json_stems(folder_path, recursive=True):
    folder = Path(folder_path)
    files = folder.rglob("*.json") if recursive else folder.glob("*.json")
    return [file.stem for file in files]

# 三个视频文件夹里的 mp4 名字（不带后缀）
folder1_list = get_mp4_stems("/mnt/mydisk/download_youtube_video/downloaded_video_surglavi")
folder2_list = get_mp4_stems("/mnt/mydisk/download_youtube_video/downloaded_video")
folder3_list = get_mp4_stems("/mnt/mydisk/download_youtube_video/downloaded_video_l")

mp4_names = set(folder1_list) | set(folder2_list) | set(folder3_list)

# json 文件名（不带后缀）
json_names = get_json_stems("/mnt/mydisk/Surglavi_videos/surglavi_narration_text")

# 检查每个 json 是否以前缀匹配某个 mp4
matched_json = []
unmatched_json = []

for json_name in json_names:
    if any(json_name.startswith(mp4_name) for mp4_name in mp4_names):
        matched_json.append(json_name)
    else:
        unmatched_json.append(json_name)

print("mp4总数:", len(mp4_names))
print("json总数:", len(json_names))
print("能在三个mp4文件夹中找到前缀匹配的json数:", len(matched_json))
print("找不到对应mp4前缀的json数:", len(unmatched_json))

print("\n前10个找不到对应mp4的json:")
print(unmatched_json[:10])

mp4总数: 22313
json总数: 3148
能在三个mp4文件夹中找到前缀匹配的json数: 2606
找不到对应mp4前缀的json数: 542

前10个找不到对应mp4的json:
['QJnJ8JVplaM', 'wKYIoT9_Aig', 'H1nrOyJjTeM', 'tFo1R4do0gA', 'UKmnYLjZmGU', 'uL6lYILBeOE', 'YH5CWLZ1DmU', 'zIWNA03WxrM', 'a9QjxojPyfs', 'iBzZsuXui5Q']


In [ ]:
import argparse
import csv
import sqlite3
from pathlib import Path


QUERY = """
SELECT
    v.youtube_id AS youtube_id,
    l.level_name AS level_name,
    c.start_time AS start_time,
    c.end_time AS end_time,
    COALESCE(NULLIF(TRIM(c.caption), ''), c.raw_narration) AS text
FROM captions AS c
JOIN videos AS v
    ON v.video_id = c.video_id
JOIN levels AS l
    ON l.level_id = c.level_id
WHERE v.youtube_id IS NOT NULL
  AND TRIM(v.youtube_id) <> ''
  AND c.is_surgical_content = 1
  AND c.is_descriptive = 1
ORDER BY v.youtube_id, c.level_id, c.start_time, c.caption_id;
"""


def parse_args() -> argparse.Namespace:
    base_dir = Path("/data/clip")

    parser = argparse.ArgumentParser(
        description="Export SurgLavi annotations, skipping videos without matching mp4 files."
    )
    parser.add_argument(
        "--db-path",
        type=Path,
        default=base_dir / "surglavi_beta.db",
        help="Path to the SQLite database file.",
    )
    parser.add_argument(
        "--output-dir",
        type=Path,
        default=base_dir / "surglavi_level_csv",
        help="Directory where fine/mid/coarse CSV folders will be written.",
    )
    parser.add_argument(
        "--missing-csv",
        type=Path,
        default=base_dir / "missing_videos.csv",
        help="Path to save missing video IDs.",
    )
    return parser.parse_args(args=[])


def collect_mp4_ids(video_roots: list[str]) -> set[str]:
    ids = set()

    for video_root in video_roots:
        root = Path(video_root)
        if not root.exists():
            print(f"warning: video directory not found: {root}")
            continue

        ids.update(mp4_file.stem for mp4_file in root.rglob("*.mp4"))

    return ids


def format_timestamp(value: float) -> str:
    return f"{value:.3f}".rstrip("0").rstrip(".")


def open_writer(output_dir: Path, level_name: str, youtube_id: str):
    level_dir = output_dir / level_name
    level_dir.mkdir(parents=True, exist_ok=True)

    csv_path = level_dir / f"{youtube_id}.csv"
    handle = csv_path.open("w", newline="", encoding="utf-8")
    writer = csv.writer(handle)
    writer.writerow(["start", "end", "text"])
    return handle, writer


def save_missing_csv(missing_ids: set[str], output_csv: Path) -> None:
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    with output_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["video_id"])
        for video_id in sorted(missing_ids):
            writer.writerow([video_id])


def export_annotations(
    db_path: Path,
    output_dir: Path,
    missing_csv: Path,
    available_mp4_ids: set[str],
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    connection = sqlite3.connect(db_path)
    connection.row_factory = sqlite3.Row

    current_key = None
    current_handle = None
    current_writer = None

    files_written = 0
    rows_written = 0
    exported_videos = set()
    annotation_videos = set()
    level_counts = {"fine": 0, "mid": 0, "coarse": 0}

    try:
        cursor = connection.execute(QUERY)

        for row in cursor:
            youtube_id = row["youtube_id"]
            level_name = row["level_name"]
            annotation_videos.add(youtube_id)

            if youtube_id not in available_mp4_ids:
                continue

            key = (youtube_id, level_name)

            if key != current_key:
                if current_handle is not None:
                    current_handle.close()

                current_handle, current_writer = open_writer(
                    output_dir=output_dir,
                    level_name=level_name,
                    youtube_id=youtube_id,
                )
                current_key = key
                files_written += 1
                exported_videos.add(youtube_id)
                level_counts[level_name] = level_counts.get(level_name, 0) + 1

            current_writer.writerow(
                [
                    format_timestamp(row["start_time"]),
                    format_timestamp(row["end_time"]),
                    row["text"],
                ]
            )
            rows_written += 1

    finally:
        if current_handle is not None:
            current_handle.close()
        connection.close()

    missing_ids = annotation_videos - available_mp4_ids
    save_missing_csv(missing_ids, missing_csv)

    print(f"Database: {db_path}")
    print(f"Output directory: {output_dir}")
    print(f"Available mp4 ids: {len(available_mp4_ids)}")
    print(f"Annotation videos in DB: {len(annotation_videos)}")
    print(f"Exported videos: {len(exported_videos)}")
    print(f"Missing videos skipped: {len(missing_ids)}")
    print(f"CSV files written: {files_written}")
    print(f"Rows written: {rows_written}")
    print(f"Missing CSV saved to: {missing_csv}")

    for level_name in ["fine", "mid", "coarse"]:
        print(f"{level_name}: {level_counts.get(level_name, 0)} files")


def main() -> None:
    args = parse_args()

    video_roots = [
        "/data/nfs_data/download_youtube_video/downloaded_video_surglavi",
        "/data/nfs_data/CLIP/downloaded_video_224_test",
    ]

    available_mp4_ids = collect_mp4_ids(video_roots)

    export_annotations(
        db_path=args.db_path.resolve(),
        output_dir=args.output_dir.resolve(),
        missing_csv=args.missing_csv.resolve(),
        available_mp4_ids=available_mp4_ids,
    )


main()


OperationalError: no such table: captions